In [1]:
# Importing Libraries
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [7]:
#Loading Tensors
x_train_tensor = torch.load("preprocessed_data/x_train_tensor.pt")
x_test_tensor  = torch.load("preprocessed_data/x_test_tensor.pt")
y_train_tensor = torch.load("preprocessed_data/y_train_tensor.pt")
y_test_tensor  = torch.load("preprocessed_data/y_test_tensor.pt")

In [17]:
#Loading Preprocessed Data
y_test  = pd.read_csv('preprocessed_data/y_test.csv').squeeze()


In [8]:
train_dataset=TensorDataset(x_train_tensor, y_train_tensor)
test_dataset=TensorDataset(x_test_tensor, y_test_tensor)

train_loader=DataLoader(train_dataset,batch_size=512, shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=512, shuffle=False)

In [12]:
#Building the neural network
model=nn.Sequential(
    nn.Linear(4,64),
    nn.ReLU(),
    nn.Dropout(0.30),
    nn.Linear(64,32),   
    nn.ReLU(),
    nn.Dropout(0.30),
    nn.Linear(32,5)
)
print(model)

Sequential(
  (0): Linear(in_features=4, out_features=64, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=64, out_features=32, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.3, inplace=False)
  (6): Linear(in_features=32, out_features=5, bias=True)
)


In [13]:
#Loss Function and Optimizer

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
#Training the Model
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                    
        predictions = model(X_batch)                
        loss        = criterion(predictions, y_batch) 
        loss.backward()                             
        optimizer.step()                         
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}  →  Loss: {avg_loss:.4f}")

Epoch 1/10  →  Loss: 0.2209
Epoch 2/10  →  Loss: 0.0577
Epoch 3/10  →  Loss: 0.0352
Epoch 4/10  →  Loss: 0.0259
Epoch 5/10  →  Loss: 0.0214
Epoch 6/10  →  Loss: 0.0188
Epoch 7/10  →  Loss: 0.0172
Epoch 8/10  →  Loss: 0.0162
Epoch 9/10  →  Loss: 0.0151
Epoch 10/10  →  Loss: 0.0142


In [ ]:
# Making Predictions
model.eval()
all_preds = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        preds   = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.numpy())

y_pred_pytorch = np.array(all_preds)

In [24]:
#Accuracy Score
print("Accuracy Score for pytorch:",round(accuracy_score(y_test, y_pred_pytorch)*100,2))

Accuracy Score for pytorch: 99.74


In [23]:
#Classification report
print("Classification report for pytorch model")
print(classification_report(y_test,y_pred_pytorch))

Classification report for pytorch model
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    109729
           1       1.00      1.00      1.00     51635
           2       1.00      0.99      1.00     28396
           3       0.99      0.99      0.99      8999
           4       0.91      1.00      0.95      1241

    accuracy                           1.00    200000
   macro avg       0.98      1.00      0.99    200000
weighted avg       1.00      1.00      1.00    200000



In [25]:
# Saving PyTorch Model
torch.save(model.state_dict(), 'saved_models/pytorch_model.pth')
print("PyTorch model saved!")

PyTorch model saved!
